# 01 · A tiny GPT forward pass in NumPy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/inference/01_tiny_gpt_forward.ipynb) &nbsp; 📖 Concept deck: [The inference workload](https://nilesh-salpe.github.io/visual-ai/inference/01-inference-workload/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** layer norm, causal multi-head self-attention, and a full decoder-only forward pass, which the other notebooks build on.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def init_params(vocab=64, d_model=32, n_heads=4, n_layers=2, max_len=64, seed=0):
    """A tiny GPT-2-style model with random weights (pre-norm blocks, tied embeddings)."""
    r = np.random.default_rng(seed)
    n = lambda *s: r.normal(0, 0.2, s)
    layers = [dict(ln1_g=1 + n(d_model) * .1, ln1_b=n(d_model) * .1, Wqkv=n(d_model, 3 * d_model), bqkv=n(3 * d_model) * .1,
                   Wo=n(d_model, d_model), bo=n(d_model) * .1, ln2_g=1 + n(d_model) * .1, ln2_b=n(d_model) * .1,
                   W1=n(d_model, 4 * d_model), b1=n(4 * d_model) * .1, W2=n(4 * d_model, d_model), b2=n(d_model) * .1)
              for _ in range(n_layers)]
    return dict(wte=n(vocab, d_model), wpe=n(max_len, d_model), layers=layers,
                lnf_g=1 + n(d_model) * .1, lnf_b=n(d_model) * .1, n_heads=n_heads)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x ** 3)))

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _ln(x, g, b, eps=1e-5):
    m = x.mean(-1, keepdims=True); v = x.var(-1, keepdims=True)
    return (x - m) / np.sqrt(v + eps) * g + b

def _attn(x, L, H):
    T, d = x.shape; dh = d // H
    q, k, v = np.split(x @ L['Wqkv'] + L['bqkv'], 3, axis=-1)
    q, k, v = [a.reshape(T, H, dh).transpose(1, 0, 2) for a in (q, k, v)]
    s = q @ k.transpose(0, 2, 1) / np.sqrt(dh)
    s = np.where(np.tril(np.ones((T, T), bool)), s, -np.inf)
    o = softmax(s) @ v
    return o.transpose(1, 0, 2).reshape(T, d) @ L['Wo'] + L['bo']

def _gpt(P, tokens):
    T = len(tokens); x = P['wte'][tokens] + P['wpe'][:T]
    for L in P['layers']:
        x = x + _attn(_ln(x, L['ln1_g'], L['ln1_b']), L, P['n_heads'])
        x = x + gelu(_ln(x, L['ln2_g'], L['ln2_b']) @ L['W1'] + L['b1']) @ L['W2'] + L['b2']
    return _ln(x, P['lnf_g'], P['lnf_b']) @ P['wte'].T

P = init_params()
print('tiny GPT ready:', len(P['layers']), 'layers, d_model =', P['wte'].shape[1], ', vocab =', P['wte'].shape[0])

## Task 1: Layer normalization

Normalize each row (token) over its features: $\mathrm{LN}(x) = g \odot \frac{x-\mu}{\sqrt{\sigma^2+\epsilon}} + b$, with $\mu,\sigma^2$ taken over the **last** axis.

In [ ]:
def layer_norm(x, g, b, eps=1e-5):
    """x: [T, d]; g, b: [d]. Returns [T, d]."""
    # TODO: mean and variance over the last axis, normalize, then scale by g and shift by b
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_layer_norm(f):
    r = np.random.default_rng(1); x = r.normal(2, 3, (5, 8)); g = r.normal(1, .1, 8); b = r.normal(0, .1, 8)
    y = f(x, g, b)
    assert y.shape == x.shape, f"shape {y.shape} != {x.shape}"
    assert np.allclose(y, _ln(x, g, b), atol=1e-6), "values differ from reference (normalize over the LAST axis, use variance not std inside the sqrt)"

check("layer_norm", _test_layer_norm, layer_norm)

## Task 2: Causal multi-head self-attention

Given the normalized input `x` ([T, d]) and one layer's weights:
1. `q, k, v = split(x @ Wqkv + bqkv)` into 3 parts of width d.
2. Reshape each into `H` heads of size `dh = d // H`, giving [H, T, dh].
3. Scores $= qk^\top/\sqrt{dh}$. **Mask** future positions (column > row) with $-\infty$, softmax over the last axis, then multiply by `v`.
4. Merge the heads back to [T, d] and apply `@ Wo + bo`.

In [ ]:
def causal_self_attention(x, layer, n_heads):
    """x: [T, d] (already layer-normed). layer: dict with Wqkv, bqkv, Wo, bo. Returns [T, d]."""
    # TODO: project to q, k, v -> split heads -> masked softmax(q k^T / sqrt(dh)) v -> merge heads -> output projection
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_causal_self_attention(f):
    x = np.random.default_rng(2).normal(size=(7, 32)); L = P['layers'][0]
    y = f(x, L, 4)
    assert y.shape == (7, 32), f"shape {y.shape} != (7, 32)"
    assert np.allclose(y, _attn(x, L, 4), atol=1e-6), "values differ from reference: check head split/merge order and the 1/sqrt(dh) scale"
    x2 = x.copy(); x2[-1] += 5.0
    assert np.allclose(f(x2, L, 4)[:-1], y[:-1], atol=1e-9), "not causal: changing the LAST token changed earlier outputs"

check("causal_self_attention", _test_causal_self_attention, causal_self_attention)

## Task 3: The full forward pass

$x = W_{te}[\text{tokens}] + W_{pe}[0..T)$, then for each layer (pre-norm):
$x \leftarrow x + \mathrm{Attn}(\mathrm{LN}_1(x))$ and $x \leftarrow x + \mathrm{GELU}(\mathrm{LN}_2(x) W_1 + b_1) W_2 + b_2$.
Finally apply $\mathrm{LN}_f$ and **tied** output weights: logits $= x\,W_{te}^\top$ ([T, vocab]).
Use your own `layer_norm` and `causal_self_attention`, or the `_solution` versions if you skipped them.

In [ ]:
def gpt_forward(params, tokens):
    """tokens: list[int] of length T. Returns logits [T, vocab]."""
    # TODO: embeddings + positions -> for each layer: x += attn(ln1(x)); x += mlp(ln2(x)) -> final ln -> x @ wte.T
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_gpt_forward(f):
    toks = [3, 14, 15, 9, 2, 6, 5]
    y = f(P, toks)
    assert y.shape == (7, 64), f"shape {y.shape} != (7, 64)"
    assert np.allclose(y, _gpt(P, toks), atol=1e-5), "logits differ from reference: check pre-norm residuals, GELU MLP, final LN and tied output weights"

check("gpt_forward", _test_gpt_forward, gpt_forward)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Layer normalization
def layer_norm_solution(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * g + b

check("layer_norm (reference)", _test_layer_norm, layer_norm_solution)

In [ ]:
# @title Solution for Task 2: Causal multi-head self-attention
def causal_self_attention_solution(x, layer, n_heads):
    T, d = x.shape; dh = d // n_heads
    q, k, v = np.split(x @ layer['Wqkv'] + layer['bqkv'], 3, axis=-1)
    q, k, v = [a.reshape(T, n_heads, dh).transpose(1, 0, 2) for a in (q, k, v)]   # [H, T, dh]
    scores = q @ k.transpose(0, 2, 1) / np.sqrt(dh)                              # [H, T, T]
    mask = np.tril(np.ones((T, T), dtype=bool))
    scores = np.where(mask, scores, -np.inf)
    out = softmax(scores, axis=-1) @ v                                             # [H, T, dh]
    return out.transpose(1, 0, 2).reshape(T, d) @ layer['Wo'] + layer['bo']

check("causal_self_attention (reference)", _test_causal_self_attention, causal_self_attention_solution)

In [ ]:
# @title Solution for Task 3: The full forward pass
def gpt_forward_solution(params, tokens):
    T = len(tokens)
    x = params['wte'][tokens] + params['wpe'][:T]
    for L in params['layers']:
        x = x + causal_self_attention_solution(layer_norm_solution(x, L['ln1_g'], L['ln1_b']), L, params['n_heads'])
        h = layer_norm_solution(x, L['ln2_g'], L['ln2_b'])
        x = x + gelu(h @ L['W1'] + L['b1']) @ L['W2'] + L['b2']
    x = layer_norm_solution(x, params['lnf_g'], params['lnf_b'])
    return x @ params['wte'].T

check("gpt_forward (reference)", _test_gpt_forward, gpt_forward_solution)

---
## Experiments
These use the reference solutions, so they run even before you finish.

In [ ]:
# Greedy generation with the full forward pass: every new token re-runs the whole prefix.
toks = [1, 2, 3]
for _ in range(10):
    toks.append(int(gpt_forward_solution(P, toks)[-1].argmax()))
print('greedy tokens:', toks)
print('note: generating token t recomputes attention over all t previous tokens; notebook 02 removes that waste with a KV cache.')

---
## Takeaways
- A decoder-only forward pass is embeddings → N × (pre-norm attention + MLP with residuals) → final norm → logits.
- The **causal mask** is what makes the model autoregressive: position *t* never sees tokens after *t*.
- Generating with the full forward pass repeats work for the whole prefix at every step. That's the waste the KV cache removes.